<center><b>CTS2000: Scripting for the Humanities</b></center>
<center>7-2: Historical Named Entity Recognition with Stanza</center>

---

# Named Entity Recognition with Stanza

## Overview

First we started with one letter and then we scaled our text analysis tools to the whole Montagu corpus, and along the way we ran into POS tagging and lemmatization. This notebook introduces a related but different task: **Named Entity Recognition (NER)**.

Where POS tagging asks "what part of speech is this word?", NER asks "does this word refer to a person, a place, an organization, a date, or something else in the world?"

We will follow the same pattern as before: start with one letter, understand the tool deeply, then scale up to the full corpus. New to this notebook:

1. **Setup**: installing and loading Stanza
2. **NER on a single sentence**: understanding entity types and the output format
3. **NER on Letter 35**: applying it to a full letter
4. **A key limitation**: why a model trained on modern news text struggles with 18th-century letters
5. **Scaling to the corpus**: running NER on all letters and storing results
6. **Aggregating entities**: counting people and places across the corpus

---

**Dataset:** Lady Mary Wortley Montagu's *Turkish Embassy Letters* (1716–1718)

---


## Step 1: Setup

[Stanza](https://stanfordnlp.github.io/stanza/ner.html) is a different library from NLTK, so it has its own installation and download steps. Unlike NLTK, where we downloaded small individual resources (`punkt`, `stopwords`, etc.), Stanza downloads a full **pipeline** which is a bundle of neural network models that work together (tokenizer, POS tagger, NER tagger, and so on).

[PyTorch](https://pypi.org/project/torch/) is a machine learning framework for Python used primarily for deep learning applications like computer vision and natural language processing. It is one of the [most important deep learning frameworks](https://pytorch.org/) today.



In [1]:
import torch
# Stanza's official models need weights_only=False on torch 2.2 
_torch_load = torch.load
torch.load = lambda *args, **kwargs: _torch_load(*args, **{**kwargs, "weights_only": False})

import stanza
import pandas as pd
import re
from collections import Counter
from itertools import chain


/Users/mervetekgurler/anaconda3/envs/cts2000/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
stanza.download('en') # download English model

2026-11-16 13:33:21 INFO: Downloaded file to /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources/resources.json
2026-11-16 13:33:21 INFO: Downloading default packages for language: en (English) ...
2026-11-16 13:33:23 INFO: File exists: /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources/en/default.zip
2026-11-16 13:33:28 INFO: Finished downloading models and saved to /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources


[['zip', 'default.zip']]

In [3]:
# Let's test the example from the repository itself
# https://github.com/stanfordnlp/stanza
# This allows us to test that the code is working as intended.

nlp = stanza.Pipeline('en') # This sets up a default neural pipeline in English
doc = nlp("Barack Obama was born in Hawaii. He was elected president in 2008.")
doc.sentences[0].print_dependencies()

2026-11-16 13:33:31 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-11-16 13:33:31 INFO: Downloaded file to /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources/resources.json
2026-11-16 13:33:33 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined_nocharlm         |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

2026-11-16 13:33:33 INFO: Using device: cpu
2026-11-16 13:33:33 INFO: Loading: tokenize
2026-11-16 13:33:34 INFO: Loading: mwt
2026-11-16 1

('Barack', 4, 'nsubj:pass')
('Obama', 1, 'flat')
('was', 4, 'aux:pass')
('born', 0, 'root')
('in', 6, 'case')
('Hawaii', 4, 'obl')
('.', 4, 'punct')


> **What just happened?** `stanza.Pipeline(...)` loaded the models into memory and built a function-like object, `nlp`, that we can now call on any piece of text. This loading step is the slow part — once `nlp` exists, processing text with it is much faster. This is why we build the pipeline once, outside of any loop, rather than rebuilding it for every letter.

In [4]:
print(doc)

[
  [
    {
      "id": 1,
      "text": "Barack",
      "lemma": "Barack",
      "upos": "PROPN",
      "xpos": "NNP",
      "feats": "Number=Sing",
      "head": 4,
      "deprel": "nsubj:pass",
      "start_char": 0,
      "end_char": 6,
      "ner": "B-PERSON",
      "multi_ner": [
        "B-PERSON"
      ]
    },
    {
      "id": 2,
      "text": "Obama",
      "lemma": "Obama",
      "upos": "PROPN",
      "xpos": "NNP",
      "feats": "Number=Sing",
      "head": 1,
      "deprel": "flat",
      "start_char": 7,
      "end_char": 12,
      "ner": "E-PERSON",
      "multi_ner": [
        "E-PERSON"
      ]
    },
    {
      "id": 3,
      "text": "was",
      "lemma": "be",
      "upos": "AUX",
      "xpos": "VBD",
      "feats": "Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin",
      "head": 4,
      "deprel": "aux:pass",
      "start_char": 13,
      "end_char": 16,
      "ner": "O",
      "multi_ner": [
        "O"
      ]
    },
    {
      "id": 4,
      "text": "b

1. There are two sentences in this example, even though we gave Stanza a single string. First, Stanza **split the text into sentences**:
    - Sentence 1: Barack Obama was born in Hawaii.
    - Sentence 2: He was elected president in 2008.

2. Each sentence was then **tokenized**: split into words and punctuation. Each token gets an `id` (its position in the sentence), its `text`, and `start_char` and `end_char`, which record where it sits in the original string.

    Next comes the **mwt** (multi-word token) step. Some tokens are really two words fused together, like *wasn't* (*was* + *not*) or *I'm* (*I* + *am*). The mwt step splits these apart. Our example sentences have none, so this step didn't change anything.

    Here is the first token of the first sentence, with everything Stanza recorded about it:

    ```json
    "id": 1,
    "text": "Barack",
    "lemma": "Barack",
    "upos": "PROPN",
    "xpos": "NNP",
    "feats": "Number=Sing",
    "head": 4,
    "deprel": "nsubj:pass",
    "start_char": 0,
    "end_char": 6,
    "ner": "B-PERSON",
    "multi_ner": [
        "B-PERSON"
    ]
    ```

    The rest of the steps each add a few of these fields.

3. The tokens were **pos**-tagged. Stanza gives two kinds of part-of-speech tag, and a list of grammatical features:

    ```json
    "upos": "PROPN",
    "xpos": "NNP",
    "feats": "Number=Sing",
    ```

    - `upos` is the **universal** POS tag. It uses the same small set of tags for every language. `PROPN` means proper noun.
    - `xpos` is the **language-specific** tag. For English, these are the same Penn Treebank tags we saw with NLTK's `pos_tag`. `NNP` means singular proper noun.
    - `feats` lists grammatical **features**, such as number, tense or person. *Barack* is singular. Compare *was* (id:3), which has five: `Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin`.

4. The tokens were **lemma**tized.

    ```json
    "lemma": "Barack",
    ```

    A name is already in its base form, so the lemma is the same as the text. The verbs are more interesting: the lemma of *born* (id:4) is *bear*, and the lemma of *was* (id:3) is *be*.

5. The sentences were **dependency parsed** (**depparse**). This step works out how the words relate to each other grammatically.

    ```json
    "head": 4,
    "deprel": "nsubj:pass",
    ```

    - `head` is the `id` of the word this one depends on. Token 4 is *born*.
    - `deprel` is the relation between them. `nsubj:pass` means *Barack* is the passive subject of *born*: the person the being-born happened to.

    Every word has exactly one head, except the main word of the sentence, the **root**. *Born* (id:4) is the root of sentence 1, so its head is `0`.

6. Finally, Stanza ran **named entity recognition** (**ner**).

    ```json
    "ner": "B-PERSON",
    "multi_ner": [
        "B-PERSON"
    ]
    ```

    The tag has two parts. `PERSON` is the entity type. The letter before it says where this token sits in the entity:

    - `B`: the **beginning** of a multi-word entity
    - `I`: **inside** one
    - `E`: the **end** of one
    - `S`: a **single**-token entity
    - `O`: **outside** any entity

    So *Barack* is `B-PERSON` and *Obama* is `E-PERSON`: together, one person. *Hawaii* is `S-GPE`, a one-word geopolitical entity.

    `multi_ner` matters only if you run several NER models at once. With one model, it just repeats `ner`.

Let's take a closer look to these sentences:

![img](../../img/stanza_sentence_1_all_layers.png)

![img](../../img/stanza_sentence_2.png)

In [ ]:
# processors='tokenize,ner' loads only what we need:
# - tokenize: splits text into words and sentences (NER depends on this)
# - ner: the named entity recognizer itself

# We could also run this instead of all the stanza models to only load the tokenizer, which we need to be able to run the NER and the NER model

ner = stanza.Pipeline(lang='en', processors='tokenize,ner')

---
## Step 2: NER on a single sentence

Before we turn to a the letters, let's pick a sentence from one letter and run NER on it

In [5]:
# it is a sentence from letter 32
sample = "My side-saddle is the first that was ever seen in this part of the world, and is gazed at with as much wonder as the ship of Columbus in the first discovery of America."

doc_sample = nlp(sample)

for sentence in doc_sample.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}") #!r:25 makes print statement look neat like a table

'first'                   ORDINAL
'Columbus'                GPE
'first'                   ORDINAL
'America'                 GPE


Let's unpack this:

`doc_sample = nlp(sample)` -> running our pipeline on a string returns a `Document` object, not a plain string or list. This is different from NLTK, where most functions returned lists or tuples directly.

`doc_sample.sentences` -> a `Document` is broken into `Sentence` objects, even if your input is one sentence. This matters for the corpus-level work later, since Stanza's own sentence splitting can disagree with NLTK's `sent_tokenize()`.

`sentence.ents` -> each `Sentence` has an `.ents` attribute: a list of entities Stanza found in it.

`ent.text` and `ent.type` -> each entity has the original text span (`ent.text`) and a category label (`ent.type`), such as `PERSON`, `GPE` (geo-political entity — countries, cities, states), `ORG`, or `DATE`.

In [6]:
# this print statement from stanza documentation shows us each word in the sentence and how it was/was not tagged

print(*[f'token: {token.text}\tner: {token.ner}' for sent in doc_sample.sentences for token in sent.tokens], sep='\n')

token: My	ner: O
token: side	ner: O
token: -	ner: O
token: saddle	ner: O
token: is	ner: O
token: the	ner: O
token: first	ner: S-ORDINAL
token: that	ner: O
token: was	ner: O
token: ever	ner: O
token: seen	ner: O
token: in	ner: O
token: this	ner: O
token: part	ner: O
token: of	ner: O
token: the	ner: O
token: world	ner: O
token: ,	ner: O
token: and	ner: O
token: is	ner: O
token: gazed	ner: O
token: at	ner: O
token: with	ner: O
token: as	ner: O
token: much	ner: O
token: wonder	ner: O
token: as	ner: O
token: the	ner: O
token: ship	ner: O
token: of	ner: O
token: Columbus	ner: S-GPE
token: in	ner: O
token: the	ner: O
token: first	ner: S-ORDINAL
token: discovery	ner: O
token: of	ner: O
token: America	ner: S-GPE
token: .	ner: O


### Entity types you'll commonly see

| Type | Meaning | Example |
|---|---|---|
| `PERSON` | People, including fictional | *Obama, Montagu* |
| `GPE` | Countries, cities, states | *Hawaii, Constantinople* |
| `LOC` | Non-GPE locations: mountains, bodies of water, regions | *the Bosphorus* |
| `ORG` | Companies, agencies, institutions | *the Senate* |
| `DATE` | Absolute or relative dates | *1717, last Tuesday* |
| `NORP` | Nationalities, religious or political groups | *French, Catholic* |
| `FAC` | Buildings, airports, bridges | *the Globe Theatre* |


> Stanza's English NER model is trained on the **[OntoNotes](https://catalog.ldc.upenn.edu/LDC2013T19)** dataset, a large collection of modern news, talk shows, web text, and similar sources. They have 18 categories for entities, which you can read more about [here on page 21](https://catalog.ldc.upenn.edu/docs/LDC2013T19/OntoNotes-Release-5.0.pdf)

### In-class exercise on manually annotating letters

In [7]:
# group 1
sample_1 = "I live in a place, that very well represents the tower of Babel: in Pera they speak Turkish, Greek, Hebrew, Armenian, Arabic, Persian, Russian, Sclavonian, Walachian, German, Dutch, French, English, Italian, Hungarian; and, what is worse, there are ten of these languages spoken in my own family.  My grooms are Arabs; my footmen French, English, and Germans; my nurse an Armenian; my house-maids Russians; half a dozen other servants, Greeks; my steward an Italian; my janizaries Turks"

doc_sample_1 = nlp(sample_1)

for sentence in doc_sample_1.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

'Babel'                   PERSON
'Pera'                    GPE
'Turkish'                 NORP
'Greek'                   NORP
'Hebrew'                  NORP
'Armenian'                NORP
'Arabic'                  LANGUAGE
'Persian'                 NORP
'Russian'                 NORP
'Sclavonian'              NORP
'Walachian'               NORP
'German'                  NORP
'Dutch'                   NORP
'French'                  NORP
'English'                 LANGUAGE
'Italian'                 NORP
'Hungarian'               NORP
'ten'                     CARDINAL
'Arabs'                   NORP
'French'                  LANGUAGE
'English'                 LANGUAGE
'Germans'                 NORP
'Armenian'                NORP
'Russians'                NORP
'half a dozen'            CARDINAL
'Greeks'                  NORP
'Italian'                 NORP
'Turks'                   NORP


In [9]:
# group 2
sample_2 = "A CERTAIN French author says, Constantinople is twice as big as Paris.  Mr W----y is unwilling to own 'tis bigger than London, though I confess it appears to me to be so; but I don't believe it is so populous."

doc_sample_2 = nlp(sample_2)

for sentence in doc_sample_2.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

'French'                  NORP
'Constantinople'          PERSON
'Paris'                   GPE
'W----y'                  PERSON
'London'                  GPE


In [10]:
# group 3
sample_3 = "I have got for you, as you desire, a Turkish love-letter, which I have put into a little box, and ordered the captain of the Smyrniote to deliver it to you with this letter."

doc_sample_3 = nlp(sample_3)

for sentence in doc_sample_3.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

'Turkish'                 NORP
'Smyrniote'               PERSON


In [11]:
# group 4
sample_4 = "The Genoese were once masters of several islands in the Archipelago, and all that part of Constantinople which is now called Galata.  Their betraying the Christian cause, by facilitating the taking of Constantinople by the Turk, deserved what has since happened to them, even the loss of all their conquests on that side to those infidels."

doc_sample_4 = nlp(sample_4)

for sentence in doc_sample_4.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

'Genoese'                 NORP
'Archipelago'             LOC
'Constantinople'          GPE
'Galata'                  GPE
'Christian'               NORP
'Constantinople'          NORP
'Turk'                    PERSON


In [12]:
# group 5
sample_5 = "I was particularly pleased with a Lucretia in the house of Balbi; the expressive beauty of that face and bosom, gives all the passion of pity and adoration, that could be raised in the soul, by the finest artist on that subject."

doc_sample_5 = nlp(sample_5)

for sentence in doc_sample_5.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

'Lucretia'                PERSON
'Balbi'                   PERSON


In [13]:
# group 6
sample_6 = "They shewed me a saint's body, where I threw a piece of money; and a picture of the virgin Mary, drawn by the hand of St Luke, very little to the credit of his painting; but, however, the finest Madona (sic) of Italy, is not more famous for her miracles."

doc_sample_6 = nlp(sample_6)

for sentence in doc_sample_6.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

'Mary'                    PERSON
'St Luke'                 PERSON
'Madona'                  PERSON
'Italy'                   GPE


---
## Step 3: NER on Letter 35

Let's bring back Letter 35, the same letter we worked with closely in 2-4 and again at the start of 3-1.

In [ ]:
# Load the corpus
montagu = pd.read_csv('../../data/montagu/montagu_letters_v3.csv')
montagu = montagu[['filename', 'title', 'body']]
montagu.head()


In [ ]:
# eding the \n line breaks from the body for stanza
montagu['body'] = montagu['body'].apply(lambda text: " ".join(text.split()))

In [ ]:
letter_35 = montagu['body'].iloc[34]
print(letter_35[:300])

In [ ]:
# Run the Stanza pipeline on the full letter
doc_35 = nlp(letter_35)

In [ ]:
print(f"Number of sentences Stanza found: {len(doc_35.sentences)}")

> Compare this sentence count to the one we got from NLTK's `sent_tokenize()` (68 sentences for Letter 35). Are they the same? If not, why might two different tools disagree about where sentences begin and end?

In [ ]:
# Extract all entities from doc_35 and print them as (text, type)

letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]

for text, etype in letter_35_entities:
    print(f"{text!r:30} {etype}")

Let's unpack the list comprehension:

```python
letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]
```

This is a **nested** list comprehension — two `for` clauses instead of one. Read it left to right, but think about it inside-out:

`for sentence in doc_35.sentences` -> outer loop, over each sentence in the document

`for ent in sentence.ents` -> inner loop, over each entity within that sentence

`(ent.text, ent.type)` -> what we keep from each entity: the text span and its label

This is equivalent to:

```python
letter_35_entities = []
for sentence in doc_35.sentences:
    for ent in sentence.ents:
        letter_35_entities.append((ent.text, ent.type))
```

Both versions do the same thing. The nested comprehension is more compact once you're comfortable with it, but there's no shame in writing the longer version first and condensing it later.

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Look closely at the output above. Are there any entities that are clearly wrong — mislabeled, or not entities at all? Are there any real people or places in the letter that Stanza missed entirely?
</div>

---
## Step 4: A key limitation — why this matters for historical text

Stanza's English NER model was trained on OntoNotes: modern news wire, broadcast conversation, web text, and similar genres from the late 20th and early 21st century. Montagu was writing private letters in 1716–1718, describing a part of the world with person and place names with several competing English spellings that does not appear in the training data.

Researchers who work on **historical NER** as a subfield have documented this gap extensively. The challenges they describe consistently include archaic spelling and inconsistent orthography, named entities that no longer exist or have changed meaning, and a basic shortage of historical text to train on compared to the enormous amount of modern text available online. None of this is a flaw in Stanza specifically — it's a structural feature of how all modern NER models get trained, and it generalizes to spaCy, FLAIR, or any other off-the-shelf system you might reach for.

> This is not a reason to abandon NER for historical text. It's a reason to treat its output as a **starting point for inquiry**, not a finished answer. We need to read the output critically to ensure that we are actually we are actually capturing what is valuable and interesting to us.

In [ ]:
# Let's look at a sentence built specifically to surface this gap.
# Try swapping in a sentence or two of your own choosing from Letter 35.

test_sentence = "We lay that night at a town called Bujuk Cekmege, or Great Bridge; and the night following, at Kujuk Cekmege, or Little Bridge; in a very pleasant lodging, formerly a monastery of dervises; having before it a large court, encompassed with marble cloisters, with a good fountain in the middle."

doc_test = nlp(test_sentence)
for sentence in doc_test.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

---
## Step 5: Scaling to the corpus

Now that we have looked closely at one letter, let's apply the same pipeline to all of Montagu's letters.

> ⚠️ Running a neural pipeline on every letter in the corpus is much slower than the simple string operations. Expect this cell to take noticeably longer to run. This is normal — Stanza is doing far more computation per word than `.split()` or even NLTK's `word_tokenize()`.

In [ ]:
# Run NER on every letter and store the resulting Document object
montagu['ner_doc'] = montagu['body'].apply(lambda text: nlp(text))

It took ca. 24 minutes to run!

We've stored the full `Document` object for each letter in ner_doc, not just the entities. This mirrors what we did with `pos_tags`: keep the intermediate result around so you don't have to recompute it if you want to extract something different from it later.

Now let's pull just the entities out into their own column.

In [ ]:
def extract_entities(doc):
    """Extract all (text, type) entity pairs from a Stanza Document."""
    return [(ent.text, ent.type) for sentence in doc.sentences for ent in sentence.ents]

montagu['entities'] = montagu['ner_doc'].apply(extract_entities)
montagu[['title', 'entities']].head()

In [ ]:
montagu.info()

> Side note: We actually ran the full nlp pipeline on all letters and not the ner only pipeline. (I forgot I switched the variable names). This is not a bad thing, we just did more work and we might as well save it better for the future.

In [ ]:
# Let's extract our stanza annotations
montagu['sentences']   = montagu['ner_doc'].apply(lambda doc: [s.text for s in doc.sentences])
montagu['num_sents']   = montagu['sentences'].apply(len)
montagu['words']       = montagu['ner_doc'].apply(lambda doc: [w.text for s in doc.sentences for w in s.words])
montagu['lemmas']      = montagu['ner_doc'].apply(lambda doc: [w.lemma for s in doc.sentences for w in s.words])
montagu['pos_tags']    = montagu['ner_doc'].apply(lambda doc: [(w.text, w.upos) for s in doc.sentences for w in s.words])
montagu['xpos_tags']   = montagu['ner_doc'].apply(lambda doc: [(w.text, w.xpos) for s in doc.sentences for w in s.words])  # Penn Treebank tags, like NLTK's
montagu['morph_feats'] = montagu['ner_doc'].apply(lambda doc: [(w.text, w.feats) for s in doc.sentences for w in s.words])  # tense, number, etc.

# Dependencies: (word, relation, head word)
montagu['dependencies'] = montagu['ner_doc'].apply(
    lambda doc: [(w.text, w.deprel, s.words[w.head - 1].text if w.head > 0 else 'ROOT')
                 for s in doc.sentences for w in s.words])

# Sentiment per sentence: 0 = negative, 1 = neutral, 2 = positive
montagu['sentiment']      = montagu['ner_doc'].apply(lambda doc: [s.sentiment for s in doc.sentences])
montagu['mean_sentiment'] = montagu['sentiment'].apply(lambda x: sum(x) / len(x))

# Constituency parse trees, saved as strings
montagu['constituency'] = montagu['ner_doc'].apply(lambda doc: [str(s.constituency) for s in doc.sentences])

In [ ]:
montagu.head()

In [ ]:
# doc.to_dict() turns each Document into plain lists/dicts (tokens, lemmas, POS, deps, NER tags, etc)
# sentiment and constituency trees are not included, so we grab those separately
montagu['doc_dict'] = montagu['ner_doc'].apply(lambda doc: doc.to_dict())

# pickle keeps Python objects intact (CSV would turn lists into strings)
montagu.drop(columns=['ner_doc']).to_pickle('../../data/montagu/montagu_stanza_full.pkl')


In [ ]:
# We could read this in later with the Stanza document

# from stanza.models.common.doc import Document

# montagu = pd.read_pickle('../../data/montagu/montagu_stanza_full.pkl')
# montagu['ner_doc'] = montagu.apply(lambda row: Document(row['doc_dict'], text=row['body']), axis=1)

In [ ]:
# Save a smaller CSV to explore later. We leave out the Stanza objects and the huge nested columns.

montagu_to_save = montagu.drop(columns=['ner_doc', 'doc_dict', 'constituency'])
montagu_to_save.to_csv('../../data/montagu/montagu_letters_v4.csv', index=False)
print("Saved! Columns now:", montagu_to_save.columns.tolist())


In [28]:
# and here is how we would access our csv later

montagu = pd.read_csv("../../data/montagu/montagu_letters_v4.csv")

montagu.head()

,filename,title,body,entities,sentences,num_sents,words,lemmas,pos_tags,xpos_tags,morph_feats,dependencies,sentiment,mean_sentiment
0,letter_1.txt,﻿LETTER 1.,"I FLATTER, myself, dear sister, that I shall g...","[('two days', 'DATE'), ('Rotterdam', 'GPE'), (...","['I FLATTER, myself, dear sister, that I shall...",18,"['I', 'FLATTER', ',', 'myself', ',', 'dear', '...","['I', 'flatter', ',', 'myself', ',', 'dear', '...","[('I', 'PRON'), ('FLATTER', 'VERB'), (',', 'PU...","[('I', 'PRP'), ('FLATTER', 'VBP'), (',', ','),...","[('I', 'Case=Nom|Number=Sing|Person=1|PronType...","[('I', 'nsubj', 'FLATTER'), ('FLATTER', 'root'...","[1, 2, 1, 1, 2, 0, 1, 1, 1, 2, 1, 0, 2, 2, 2, ...",1.222222
1,letter_2.txt,LET. II,"I MAKE haste to tell you, dear Madam, that, af...","[('Madam', 'PERSON'), ('Holland', 'GPE'), ('tw...","['I MAKE haste to tell you, dear Madam, that, ...",12,"['I', 'MAKE', 'haste', 'to', 'tell', 'you', ',...","['I', 'make', 'haste', 'to', 'tell', 'you', ',...","[('I', 'PRON'), ('MAKE', 'VERB'), ('haste', 'N...","[('I', 'PRP'), ('MAKE', 'VBP'), ('haste', 'NN'...","[('I', 'Case=Nom|Number=Sing|Person=1|PronType...","[('I', 'nsubj', 'MAKE'), ('MAKE', 'root', 'ROO...","[2, 1, 2, 2, 2, 2, 2, 1, 1, 0, 1, 2]",1.500000
2,letter_3.txt,LET. III,"I AM extremely sorry, my dear S. that your fea...","[('S.', 'PERSON'), ('Nottingham', 'GPE'), ('tw...","['I AM extremely sorry, my dear S. that your f...",18,"['I', 'AM', 'extremely', 'sorry', ',', 'my', '...","['I', 'be', 'extremely', 'sorry', ',', 'my', '...","[('I', 'PRON'), ('AM', 'AUX'), ('extremely', '...","[('I', 'PRP'), ('AM', 'VBP'), ('extremely', 'R...","[('I', 'Case=Nom|Number=Sing|Person=1|PronType...","[('I', 'nsubj', 'sorry'), ('AM', 'cop', 'sorry...","[1, 1, 1, 1, 1, 2, 0, 2, 2, 2, 1, 1, 1, 1, 2, ...",1.166667
3,letter_4.txt,LET. IV.,IF my lady ---- could have any notion of the f...,"[('two last days', 'DATE'), ('Nimeguen', 'GPE'...",['IF my lady ---- could have any notion of the...,12,"['IF', 'my', 'lady', '----', 'could', 'have', ...","['if', 'my', 'lady', '----', 'could', 'have', ...","[('IF', 'SCONJ'), ('my', 'PRON'), ('lady', 'NO...","[('IF', 'IN'), ('my', 'PRP$'), ('lady', 'NN'),...","[('IF', None), ('my', 'Case=Gen|Number=Sing|Pe...","[('IF', 'mark', 'have'), ('my', 'nmod:poss', '...","[0, 1, 0, 0, 2, 0, 1, 2, 1, 1, 1, 1]",0.833333
4,letter_5.txt,LET. V.,"AFTER five days travelling post, I could not s...","[('five days', 'DATE'), ('Germany', 'GPE'), ('...","['AFTER five days travelling post, I could not...",19,"['AFTER', 'five', 'days', 'travelling', 'post'...","['after', 'five', 'day', 'travel', 'post', ','...","[('AFTER', 'ADP'), ('five', 'NUM'), ('days', '...","[('AFTER', 'IN'), ('five', 'CD'), ('days', 'NN...","[('AFTER', None), ('five', 'NumForm=Word|NumTy...","[('AFTER', 'case', 'post'), ('five', 'nummod',...","[0, 0, 1, 1, 1, 2, 0, 0, 1, 2, 0, 1, 1, 1, 1, ...",0.789474


---
## Step 6: Aggregating entities across the corpus

With entities extracted for every letter, we can ask corpus-level questions: who does Montagu mention most? What places recur across her letters?

### People

In [29]:
montagu["entities"].head()

0    [('two days', 'DATE'), ('Rotterdam', 'GPE'), (...
1    [('Madam', 'PERSON'), ('Holland', 'GPE'), ('tw...
2    [('S.', 'PERSON'), ('Nottingham', 'GPE'), ('tw...
3    [('two last days', 'DATE'), ('Nimeguen', 'GPE'...
4    [('five days', 'DATE'), ('Germany', 'GPE'), ('...
Name: entities, dtype: str

In [30]:
# Flatten all entities into one list, same pattern as chain.from_iterable in 3-1
all_entities = list(chain.from_iterable(montagu['entities']))

# Keep only PERSON entities
people = [text for text, etype in all_entities if etype == 'PERSON']
person_counts = Counter(people)

print("Top 20 most-mentioned people:")
for name, count in person_counts.most_common(20):
    print(f"  {name}: {count}")

ValueError: not enough values to unpack (expected 2, got 1)

**Why did we get this error?**

We saved our data as a csv and that coverted everything to a string. What looks like to us as a list of tuples is actually just one whole string that we converted to a list in `all_entities = list(chain.from_iterable(montagu['entities']))`.

This does not mean however that everything else got turned into the correct data types.

Don't worry, there is a fix!!

In [ ]:
type(all_entities)

In [ ]:
type(all_entities[0])

In [ ]:
print(all_entities[0])

In [ ]:
print(all_entities[:20])

`ast` is a Python module, Abstract Syntax Trees. `ast.literal_eval` takes a string of Python and turns it inot the real value it describes

In [31]:
import ast

all_entities = list(chain.from_iterable(ast.literal_eval(ents) for ents in montagu['entities']))

In [ ]:
type(all_entities)

In [ ]:
type(all_entities[0])

In [ ]:
print(all_entities[0])

In [32]:
# Keep only PERSON entities
people = [text for text, etype in all_entities if etype == 'PERSON']
person_counts = Counter(people)

print("Top 20 most-mentioned people:")
for name, count in person_counts.most_common(20):
    print(f"  {name}: {count}")

Top 20 most-mentioned people:
  W: 16
  Turk: 8
  Adieu: 7
  Homer: 6
  Fatima: 6
  I'll: 4
  Mustapha: 4
  Paul Rycaut: 4
  Constantinople: 4
  Sancta Sophia: 4
  Madam: 3
  S.: 3
  Amelia: 3
  Virgil: 3
  Helen: 3
  Constantine: 3
  Lucretia: 3
  Bartholomew: 2
  Lorrain: 2
  Amphitrion: 2


<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Scan this list. Are these all genuinely distinct people, or are they even all people?
</div>

### Places

What counts as a place for Stanza?

- GPE (geo-political entity) covers cities and countries.
- LOC covers rivers, seas and regions.

Let's extract GPE and LOC entities and count the most common ones.

In [33]:
places = [text for text, etype in all_entities if etype in ('GPE', 'LOC')]
place_counts = Counter(places)

print("Top 20 most-mentioned places:")
for place, count in place_counts.most_common(20):
    print(f"  {place}: {count}")

Top 20 most-mentioned places:
  Vienna: 33
  England: 26
  London: 25
  Constantinople: 22
  Paris: 16
  Hungary: 14
  Germany: 13
  Danube: 11
  Belgrade: 11
  France: 8
  Europe: 7
  Peterwaradin: 7
  Italy: 7
  Dresden: 6
  Asia: 6
  Turkey: 6
  Hanover: 5
  Pera: 5
  Carthage: 5
  Genoa: 4


In [34]:
# Pick one place from your list above and confirm it actually appears where you'd expect
search_term = "Vienna"  # change this to check on a new term

matches = montagu[montagu['body'].str.contains(search_term, case=False, na=False)]
print(f"'{search_term}' appears in {len(matches)} letter(s):")
print(matches['title'].tolist())

'Vienna' appears in 15 letter(s):
['LET. VII.', 'LET. IX.', 'LET. X.', 'LET. XI.', 'LET. XII.', 'LET.  XIV.', 'LET. XV.', 'LET. XVII.', 'LET. XVIII.', 'LET. XIX.', 'LET. XX.', 'LET. XXI.', 'LET. XXIII.', 'LET. XXXIII.', 'LET. XXXVII.']


People will have to wait for the next notebook but places seem good to go for a first try!

## Step 7: Can we map these places?

Yes! but it requires a few steps.

First, we need to decide what we are exactly visualizing. There are a few options.

*We have:* places mentioned in each letter.

*We can visualize:* in how many letters a given place is mentioned.

Let's get this into a more useable format.

*Step 1:* Create a new column in our df that for each letter shows the placename and its counts.

In [35]:
# remember our string problem?
montagu['entities'] = montagu['entities'].apply(ast.literal_eval)

In [36]:
montagu["places"] = montagu["entities"].apply(
    lambda ents: Counter(text for text, etype in ents if etype in ('GPE', 'LOC'))
)

montagu[["title", "places"]].head()

,title,places
0,﻿LETTER 1.,"{'Rotterdam': 3, 'Helvoetsluys': 1, 'England':..."
1,LET. II,"{'Holland': 1, 'London': 1, 'India': 1}"
2,LET. III,"{'Nottingham': 3, 'Trentfield': 1, 'Adboulton'..."
3,LET. IV.,"{'Nimeguen': 1, 'Stamel': 1, 'St Ursula's': 1,..."
4,LET. V.,"{'Germany': 2, 'Cologn': 1, 'Frankfort': 1, 'W..."


*Step 2:* Let's get a list of all the unique places

In [37]:
# did you know that you can write a comprehension split across lines instead of all in one line?
# Python lets you break a line anywhere inside brackets, so splitting it up lets each part of the comprehension sit on its own line

all_unique_places = sorted(set(
    text
    for ents in montagu['entities']
    for text, etype in ents
    if etype in ('GPE', 'LOC')
))


print(len(all_unique_places), "unique places")

205 unique places


205 unique places are a lot of places to map at once.

*Step 3:* Let's get a sense of how often these places appear

In [ ]:
# recreating place counts here independently of section 6

place_counts = Counter()

# the places column holds a counter for each letter so, it's easy mode
for c in montagu["places"]:
    place_counts.update(c)

In [39]:
# this is our counter for corpus wide place names
letter_counts = Counter(place for c in montagu["places"] for place in c.keys())

# mentioned only once
once = [p for p, n in place_counts.items() if n == 1]
# mentioned only twice
twice = [p for p, n in place_counts.items() if n == 2]
# mentioned twice in the same letter
twice_same_letter = [p for p in twice if letter_counts[p] == 1]
# mentioned in two different letters
twice_diff_letters = [p for p in twice if letter_counts[p] == 2]

print(f"Unique places in total: {len(place_counts)}")
print(f"Mentioned only once: {len(once)}")
print(f"Mentioned exactly twice: {len(twice)}")
print(f"  both in the same letter: {len(twice_same_letter)}  {sorted(twice_same_letter)}")
print(f"  in two different letters: {len(twice_diff_letters)}  {sorted(twice_diff_letters)}")

Unique places in total: 205
Mentioned only once: 142
Mentioned exactly twice: 25
  both in the same letter: 9  ['Abydos', 'Candia', 'Chersonesus', 'Chios', 'Fauxburg', 'Japan', 'Poland', 'Sigaeum', 'Versailles']
  in two different letters: 16  ['Adrianople', 'Aleppo', 'Archipelago', 'Betsko', 'Frankfort', 'Galata', 'Grand Cairo', 'Jupiter', 'Leipzig', 'Levant', 'Mecca', 'Morea', 'Prague', 'Sophia', 'Venice', 'china']


Unique places in total: 205

Mentioned only once: 142

Mentioned exactly twice: 25

  both in the same letter: 9  ['Abydos', 'Candia', 'Chersonesus', 'Chios', 'Fauxburg', 'Japan', 'Poland', 'Sigaeum', 'Versailles']
  
  in two different letters: 16  ['Adrianople', 'Aleppo', 'Archipelago', 'Betsko', 'Frankfort', 'Galata', 'Grand Cairo', 'Jupiter', 'Leipzig', 'Levant', 'Mecca', 'Morea', 'Prague', 'Sophia', 'Venice', 'china']

Interesting! Two take aways:

- 142 out of 205 places are only mentioned once.
- Perhaps there are some spelling errors? It is hard to imagine Adrianople was mentioned only in two letters.

*Step 4:* Let's investigate Adrianople

In [40]:
place = "Adrianople"

# 1. Ground truth: how often does the word appear in the raw letters?
raw_hits = montagu["body"].str.count(place)
print(f"In the raw text: {raw_hits.sum()} mentions in {(raw_hits > 0).sum()} letters")

# 2. What did Stanza call it?
labels = Counter(etype for ents in montagu["entities"] for text, etype in ents if text == place)
print("Stanza's labels:", labels)

# 3. Look for spelling variants, just in case
variants = Counter(w for body in montagu["body"] for w in re.findall(r"\b[Aa]dri\w*", body))
print("Variants in the text:", variants)


In the raw text: 8 mentions in 5 letters
Stanza's labels: Counter({'NORP': 2, 'GPE': 2, 'PERSON': 2, 'ORG': 2})
Variants in the text: Counter({'Adrianople': 8, 'Adrian': 1})


Aha! The issue was Stanza's categorization, which is to be expected with an NER model trained on modern texts.

Before we fix this, we need to run the same check on other places we expect to be common, like Constantinople, Pera or Belgrade, to see whether the mislabelling happens elsewhere. "Constantinople" and "Sancta Sophia" already showed up as PERSON top 20.

*Step 5:* Let's fix some placenames semi-automatically

In [41]:
# I did a first pass and chose these

suspect_places = ["Constantinople", "Adrianople", "Peterwaradin", "Cairo", "St Pierre d'Arena"]

for place in suspect_places:
    raw = montagu["body"].str.count(re.escape(place)).sum()
    labels = Counter(etype for ents in montagu["entities"] for text, etype in ents if text == place)
    as_place = labels["GPE"] + labels["LOC"]
    print(f"{place:20} raw text: {raw:3}   tagged: {sum(labels.values()):3}   "
          f"as GPE/LOC: {as_place:3}   labels: {dict(labels)}")


Constantinople       raw text:  35   tagged:  32   as GPE/LOC:  22   labels: {'PERSON': 4, 'GPE': 22, 'ORG': 3, 'NORP': 3}
Adrianople           raw text:   8   tagged:   8   as GPE/LOC:   2   labels: {'NORP': 2, 'GPE': 2, 'PERSON': 2, 'ORG': 2}
Peterwaradin         raw text:  10   tagged:  10   as GPE/LOC:   7   labels: {'GPE': 7, 'PERSON': 2, 'ORG': 1}
Cairo                raw text:   3   tagged:   1   as GPE/LOC:   1   labels: {'GPE': 1}
St Pierre d'Arena    raw text:   3   tagged:   3   as GPE/LOC:   1   labels: {'FAC': 2, 'GPE': 1}


In [43]:
# while working on that, I also realized that a few places were not split correctly

merge_map = {"GENOA": "Genoa", "Old England": "England", "Grand Cairo": "Cairo"}

for variant, standard in merge_map.items():
    for name in (variant, standard):
        raw = montagu["body"].str.count(re.escape(name)).sum()
        labels = Counter(etype for ents in montagu["entities"] for text, etype in ents if text == name)
        as_place = labels["GPE"] + labels["LOC"]
        arrow = "  ->" if name == standard else ""
        print(f"{arrow:4}{name:16} raw text: {raw:3}   tagged: {sum(labels.values()):3}   "
              f"as GPE/LOC: {as_place:3}   labels: {dict(labels)}")
    print()

    GENOA            raw text:   1   tagged:   1   as GPE/LOC:   1   labels: {'GPE': 1}
  ->Genoa            raw text:   4   tagged:   4   as GPE/LOC:   4   labels: {'GPE': 4}

    Old England      raw text:   1   tagged:   1   as GPE/LOC:   1   labels: {'GPE': 1}
  ->England          raw text:  27   tagged:  26   as GPE/LOC:  26   labels: {'GPE': 26}

    Grand Cairo      raw text:   2   tagged:   2   as GPE/LOC:   2   labels: {'GPE': 2}
  ->Cairo            raw text:   3   tagged:   1   as GPE/LOC:   1   labels: {'GPE': 1}



In [44]:
montagu["places_fixed"] = montagu["entities"].apply(
    lambda ents: Counter(merge_map.get(text, text) for text, etype in ents
                         if etype in ("GPE", "LOC") or text in suspect_places)
)

# Let's see what we changed
changed = montagu[montagu["places"] != montagu["places_fixed"]]
changed[["title", "places_fixed"]]

,title,places_fixed
23,LET. XXIV.,"{'Peterwaradin': 5, 'Hungary': 3, 'Cairo': 1, ..."
26,LET. XXVII.,"{'England': 2, 'Belgrade': 3, 'Servia': 1, 'Ba..."
27,LET. XXVIII,"{'London': 1, 'Constantinople': 1, 'Peterwarad..."
31,LET. XXXII.,"{'Turkey': 2, 'London': 1, 'Peterwaradin': 1, ..."
33,LET. XXXIV.,"{'Adrianople': 1, 'Constantinople': 2, 'Hebrus..."
34,LET. XXXV.,"{'Asia': 2, 'Ciorlei': 1, 'Marmora': 1, 'Propo..."
38,LET. XXXIX.,"{'England': 2, 'London': 3, 'Constantinople': ..."
40,LET. XLI.,"{'Constantinople': 6, 'Pera': 2, 'Tophana': 1,..."
41,LET. XLII.,"{'Constantinople': 5, 'Pera': 1, 'Naples': 1, ..."
44,LET. XLV,"{'Tunis': 1, 'Constantinople': 4, 'Italy': 2, ..."


In [45]:
# this is our UPDATED counter for corpus wide place mentions
place_counts_updated = sum(montagu["places_fixed"], Counter())

# this is our UPDATED counter for how many letters each place appears in
letter_counts_updated = Counter(place for c in montagu["places_fixed"] for place in c.keys())

# mentioned only once
once = [p for p, n in place_counts_updated.items() if n == 1]
# mentioned only twice
twice = [p for p, n in place_counts_updated.items() if n == 2]
# mentioned twice in the same letter
twice_same_letter = [p for p in twice if letter_counts_updated[p] == 1]
# mentioned in two different letters
twice_diff_letters = [p for p in twice if letter_counts_updated[p] == 2]

print(f"Unique places in total: {len(place_counts_updated)}")
print(f"Mentioned only once: {len(once)}")
print(f"Mentioned exactly twice: {len(twice)}")
print(f"  both in the same letter: {len(twice_same_letter)}  {sorted(twice_same_letter)}")
print(f"  in two different letters: {len(twice_diff_letters)}  {sorted(twice_diff_letters)}")


Unique places in total: 202
Mentioned only once: 138
Mentioned exactly twice: 23
  both in the same letter: 9  ['Abydos', 'Candia', 'Chersonesus', 'Chios', 'Fauxburg', 'Japan', 'Poland', 'Sigaeum', 'Versailles']
  in two different letters: 14  ['Aleppo', 'Archipelago', 'Betsko', 'Frankfort', 'Galata', 'Jupiter', 'Leipzig', 'Levant', 'Mecca', 'Morea', 'Prague', 'Sophia', 'Venice', 'china']


We could investigate this further, like `china` in lowercase looks suspicious to me because it is more likely to be referencing an object made of porcelain. We will take a closer look at this in another notebook. 

*Step 6:* Let's get a list of unique places after keeping places that appear in at least two letters and mentioned at least 3 times

In [46]:
recurring_places = sorted(
    (p for p in place_counts_updated
     if letter_counts_updated[p] >= 2 and place_counts_updated[p] >= 3),
    key=lambda p: -place_counts_updated[p]
)

print(f"{len(recurring_places)} places")

35 places


In [47]:
for p in recurring_places:
    print(f"  {p:20} {place_counts_updated[p]:3} mentions in {letter_counts_updated[p]:2} letters")

  Vienna                33 mentions in 15 letters
  Constantinople        32 mentions in 14 letters
  England               27 mentions in 21 letters
  London                25 mentions in 20 letters
  Paris                 16 mentions in  9 letters
  Hungary               14 mentions in  5 letters
  Germany               13 mentions in  9 letters
  Danube                11 mentions in  6 letters
  Belgrade              11 mentions in  6 letters
  Peterwaradin          10 mentions in  5 letters
  Adrianople             8 mentions in  5 letters
  France                 8 mentions in  7 letters
  Europe                 7 mentions in  6 letters
  Italy                  7 mentions in  6 letters
  Dresden                6 mentions in  4 letters
  Asia                   6 mentions in  5 letters
  Turkey                 6 mentions in  5 letters
  Hanover                5 mentions in  4 letters
  Genoa                  5 mentions in  4 letters
  Pera                   5 mentions in  4 letters


35 places! Very manageable and interesting!

*Step 7:* Read the letters and evaluate these places

In [48]:
def kwic(term, width=50):
    """Print every mention of term, with `width` characters of context on each side."""
    for title, body in zip(montagu["title"], montagu["body"]):
        for match in re.finditer(re.escape(term), body):
            left = body[max(0, match.start() - width):match.start()].replace("\n", " ")
            right = body[match.end():match.end() + width].replace("\n", " ")
            print(f"{title:10} {left:>{width}} [{match.group()}] {right}")

In [49]:
kwic("St Lawrence", width=100)

LET. XII.   say something of the convents; they are of all sorts and sizes, but I am best pleased with that of  [St Lawrence] , where the ease and neatness they seem to live with, appears to be much more edifying than those st
LET. XLV   had not first engaged my eyes.--Here are also some inestimable ancient bustos (sic).--The church of  [St Lawrence]  is built of black and white marble, where is kept that famous plate of a single emerald, which is n
LET. XLIX. erior marks of it at Paris, where all peoples thoughts seem to be on present diversion. THE fair of  [St Lawrence]  is now in season. You may be sure I have been carried thither, and think it much better disposed th


Let's unpack this:

`def kwic(term, width=50):` -> a function with two inputs: the word we're looking for, and how many characters of context to show on each side. `width=50` is a **default value**: if we don't give a width, Python uses 50. So `kwic("St Lawrence")` and `kwic("St Lawrence", width=50)` do the same thing, but we can also write `kwic("St Lawrence", width=100)` to see more context.

`for title, body in zip(montagu["title"], montagu["body"]):` -> `zip` pairs up the two columns, so on each loop we get one letter's title and its text together. This lets us print which letter each mention comes from.

`for match in re.finditer(re.escape(term), body):` -> `re.finditer` finds **every** match in the letter, not just the first one. Each `match` remembers where it starts and ends in the text.

`re.escape(term)` -> in regular expressions, some characters have special meanings. For example, `.` means "any character". `re.escape` tells Python to search for our term exactly as we typed it.

`match.start()` and `match.end()` -> the position of the first character of the match, and the position just after its last character. These are like the `start_char` and `end_char` that Stanza gave us for each token.

`left = body[max(0, match.start() - width):match.start()]` -> we slice out the `width` characters just before the match. If the match is near the beginning of a letter, `match.start() - width` could be a negative number, and a negative number in a slice counts from the **end** of the text. `max(0, ...)` picks whichever is larger, so the slice never starts before 0.

`right = body[match.end():match.end() + width]` -> the same for the text just after the match. We don't need `max` here, because Python simply stops at the end of the text if the slice goes past it.

`.replace("\n", " ")` -> we replace line breaks with spaces, so each mention prints on a single line.

`print(f"{title:10} {left:>{width}} [{match.group()}] {right}")` -> this is what makes it look like a KWIC:
- `{title:10}` gives the title 10 characters of space, so the columns line up.
- `{left:>{width}}` **right-aligns** the left context. This pushes every keyword into the same column, so we can read down the middle.
- `[{match.group()}]` prints the match itself in square brackets, so it stands out.

PS: KWIC stands for **KeyWord In Context**. You can use this code or a version of it for visualizing any term that you want to show in context.

In [50]:
# St Lawrence passed our filters, but reading the letters shows three different things:
# a convent in Vienna, a church in Genoa, and a fair in Paris. Not the same place mentioned three times no a place on her route.
drop = {"St Lawrence"}
recurring_places = [p for p in recurring_places if p not in drop]

print(f"{len(recurring_places)} places")

34 places


### Interlude

Remember how we identified the locations that the letters were sent from during our text extraction step? They will now come very handy!

In [51]:
locations = pd.read_csv("../../data/montagu/locations.csv")
locations.head()

,location_id,location_name,location_wikidata,latitude,longitude
0,LOC-001,Rotterdam,Q34370,51.920000,4.480000
1,LOC-002,Hague,Q36600,52.080000,4.310000
2,LOC-003,Nimeguen,Q47887,51.847500,5.862500
3,LOC-004,Cologne,Q365,50.942222,6.957778
4,LOC-005,Nuremberg,Q2090,49.453889,11.077500


In [52]:
# Montagu's spelling -> the modern name used in locations.csv
# This is from my original matching on Google Sheets. 
historical_to_modern = {
    "Constantinople": "Istanbul",
    "Adrianople": "Edirne",
    "Peterwaradin": "Petrovaradin",
    "Blankenburg": "Blankenburg am Harz",
}

In [53]:
print("Places mentioned in the letters that are also sending locations:\n")
for place in recurring_places:
    modern = historical_to_modern.get(place, place)
    match = locations[locations["location_name"] == modern]
    if not match.empty:
        row = match.iloc[0]
        print(f"  {place:16} -> {row['location_name']:20} {row['location_id']}  {row['location_wikidata']}  "
              f"mentioned {place_counts_updated[place]} times")

Places mentioned in the letters that are also sending locations:

  Vienna           -> Vienna               LOC-007  Q1741  mentioned 33 times
  Constantinople   -> Istanbul             LOC-016  Q16869  mentioned 32 times
  Paris            -> Paris                LOC-023  Q90  mentioned 16 times
  Belgrade         -> Belgrade             LOC-014  Q3711  mentioned 11 times
  Peterwaradin     -> Petrovaradin         LOC-013  Q19429  mentioned 10 times
  Adrianople       -> Edirne               LOC-015  Q43387  mentioned 8 times
  Hanover          -> Hanover              LOC-011  Q1715  mentioned 5 times
  Genoa            -> Genoa                LOC-020  Q1449  mentioned 5 times
  Pera             -> Pera                 LOC-018  Q217411  mentioned 5 times
  Tunis            -> Tunis                LOC-019  Q3572  mentioned 4 times
  Blankenburg      -> Blankenburg am Harz  LOC-012  Q525960  mentioned 3 times
  Turin            -> Turin                LOC-021  Q495  mentioned 3 times


In [54]:
# let's keep track of the ones we don't have QIDs for

not_in_locations = [p for p in recurring_places
                    if historical_to_modern.get(p, p) not in set(locations["location_name"])]
not_in_locations

['England',
 'London',
 'Hungary',
 'Germany',
 'Danube',
 'France',
 'Europe',
 'Italy',
 'Dresden',
 'Asia',
 'Turkey',
 'Bohemia',
 'Philippopolis',
 'Rome',
 'Holland',
 'Austria',
 'Alps',
 'Buda',
 'Essek',
 'Cairo',
 'Servia',
 'Hebrus']

What do you think of these places? Are they all the same kind of place? Are they all cities or countries?

**Overview**

| Type | Places |
|---|---|
| Countries and regions | England, Hungary, Germany, France, Italy, Turkey, Bohemia, Holland, Austria, *Servia* |
| Continents | Europe, Asia |
| Rivers and mountains | Danube, Hebrus, Alps |
| Cities and towns | London, Dresden, Rome, *Philippopolis*, *Buda*, *Essek*, Cairo |


Wait, what is *Servia*?

Servia is an old spelling for Serbia. Here is an example from the Broadview edition of the letters, which I used for cross-referencing while preparing these placenames:

![img](../../img/broadview_99_servia.png)

![img](../../img/broadview_99_footnote.png)

Before we can map these places, we need their coordinates. To find coordinates, we need Wikidata QIDs (or another reliable entity linking system). To search for Wikidata QIDs, we need modern placenames. Luckily, for many modern placenames are quite similar but not for all.

Find the modern names for these places:

- Philippopolis
- Buda
- Essek

In [55]:
historical_to_modern = {
    # sending locations (match locations.csv)
    "Constantinople": "Istanbul",
    "Adrianople": "Edirne",
    "Peterwaradin": "Petrovaradin",
    "Blankenburg": "Blankenburg am Harz",
    # other places mentioned in the letters
    "Servia": "Serbia",
    "Philippopolis": "Plovdiv",
    "Essek": "Osijek",
    "Hebrus": "Maritsa River",
    "Buda": "Budapest",
    "Holland": "Netherlands",
    "Turkey": "Ottoman Empire",
}

In [56]:
# a small table: Montagu's name next to the modern one
place_names = pd.DataFrame({"place": recurring_places})
place_names["modern_name"] = place_names["place"].map(lambda p: historical_to_modern.get(p, p))
place_names

,place,modern_name
0,Vienna,Vienna
1,Constantinople,Istanbul
2,England,England
3,London,London
4,Paris,Paris
5,Hungary,Hungary
6,Germany,Germany
7,Danube,Danube
8,Belgrade,Belgrade
9,Peterwaradin,Petrovaradin


In [57]:
# QIDs we already have from locations.csv
qid_lookup = dict(zip(locations["location_name"], locations["location_wikidata"]))

place_names["QID"] = place_names["modern_name"].map(qid_lookup)

place_names

,place,modern_name,QID
0,Vienna,Vienna,Q1741
1,Constantinople,Istanbul,Q16869
2,England,England,NaN
3,London,London,NaN
4,Paris,Paris,Q90
5,Hungary,Hungary,NaN
6,Germany,Germany,NaN
7,Danube,Danube,NaN
8,Belgrade,Belgrade,Q3711
9,Peterwaradin,Petrovaradin,Q19429


In [58]:
# this is how we can see the ones that don't have a QID from the locations.csv
place_names[place_names["QID"].isna()]

,place,modern_name,QID
2,England,England,NaN
3,London,London,NaN
5,Hungary,Hungary,NaN
6,Germany,Germany,NaN
7,Danube,Danube,NaN
11,France,France,NaN
12,Europe,Europe,NaN
13,Italy,Italy,NaN
14,Dresden,Dresden,NaN
15,Asia,Asia,NaN


There are two ways to go about this. One, we can look up all these places **manually on Wikidata**. Two, we can use **API** calls.

Let's do this manually first, look up a few places on wikidata: 

## Wikidata API

### What is an API?

This workshop is about APIs. You may have heard this terminology in a variety of programming settings. What exactly does it mean?

* "API" stands for **Application Programming Interface**.

* Broadly defined, an API is a set of rules and procedures that facilitate interactions between computers and their applications.

* A very common type of API is the Web API, which, among other things, allows users to query a remote database over the internet.

* For example, a web service such as Reddit has many databases that may be of use to us: Posts, Users, Subreddits, etc. If we want to access some portion of these databases, it'd be helpful to have a set of rules and protocols in place to outline how we access this information. This is the motivation for an API.

* Web APIs take on a variety of formats, but the vast majority adhere to a particular style known as **Representational State Transfer** or **REST**.

* What makes these "RESTful" APIs so convenient is that we can use them to query databases using URLs.

### RESTful Web APIs Are All Around You

Consider a simple Google search:

![](../../img/google_search.png)

Ever wonder what all that extra stuff in the address bar was all about?  In this case, the full address is Google's way of sending a query to its databases asking requesting information related to the search term "golden state warriors". 

![](../../img/google_link.png)

In fact, it looks like Google makes its query by taking the search terms, separating each of them with a "+", and appending them to the link "https://www.google.com/#q=".  Therefore, we should be able to actually change our Google search by adding some terms to the URL and following the general format:

![](../../img/google_link_change.png)

Using RESTful APIs is essentially formatting these URLs so that you can get the response you want.

### Some Terminology

* **Uniform Resource Locator (URL)**: a string of characters that, when interpreted via the Hypertext Transfer Protocol (HTTP), points to a data resource, notably files written in Hypertext Markup Language (HTML) or a subset of a database.  This is often referred to as a "call".

* **HTTP Methods/Verbs**:

    + *GET*: requests a representation of a data resource corresponding to a particular URL.  The process of executing the GET method is often referred to as a "GET request" and is the main method used for querying RESTful databases.
    
    + *HEAD*, *POST*, *PUT*, *DELETE*: other common methods, though mostly never used for database querying.
    
As you might suspect from the example above, surfing the web is basically equivalent to sending a bunch of GET requests to different servers and asking for different files written in HTML.


### API or Web Scraping?

When deciding between using an API or web scraping, you should consider both the method's legality and efficiency. APIs provide structured, authorized access to data, often with clear documentation and rate limits to manage server load.

Web scraping, on the other hand, involves extracting data from web pages, which may violate a site's terms of service or lead to challenges in navigating complex page structures.

While scraping can be useful when no API is available, APIs are generally the preferred method for accessing web data due to their reliability and compliance with legal standards.

### [Wikidata API](https://www.wikidata.org/wiki/Wikidata:REST_API)

The Wikibase REST API is an OpenAPI-based interface that allows users to interact with, retrieve and edit Items, Properties and Statements on Wikibase instances -- including of course Wikidata. For more information about REST, see the Wikipedia entry on representational state transfer.

**Let's search for one place: Plovdiv**

In [59]:
import requests

url = "https://www.wikidata.org/w/api.php"

params = {
    "action": "wbsearchentities",   # "search for items"
    "search": "Plovdiv",            # what we're looking for
    "language": "en",               # search English labels
    "format": "json",               # give us the answer as JSON
}

In [60]:
# Wikidata asks every script to say who it is
headers = {"User-Agent": "CTS2000-class-notebook/1.0"}

response = requests.get(url, params=params, headers=headers)
data = response.json()

data["search"][0]

{'id': 'Q459',
 'title': 'Q459',
 'pageid': 693,
 'concepturi': 'http://www.wikidata.org/entity/Q459',
 'repository': 'wikidata',
 'url': '//www.wikidata.org/wiki/Q459',
 'display': {'label': {'value': 'Plovdiv', 'language': 'en'},
  'description': {'value': 'city in Plovdiv municipality, Plovdiv oblast, Bulgaria',
   'language': 'en'}},
 'label': 'Plovdiv',
 'description': 'city in Plovdiv municipality, Plovdiv oblast, Bulgaria',
 'match': {'type': 'label', 'language': 'en', 'text': 'Plovdiv'}}

**What do we want to save from this?**

In [61]:
first = data["search"][0]
print(first["id"], "|", first["label"], "|", first.get("description"))

Q459 | Plovdiv | city in Plovdiv municipality, Plovdiv oblast, Bulgaria


In [64]:
# first? does this imply that there is a second item returned in our request?

data["search"][1]

{'id': 'Q187874',
 'title': 'Q187874',
 'pageid': 186596,
 'concepturi': 'http://www.wikidata.org/entity/Q187874',
 'repository': 'wikidata',
 'url': '//www.wikidata.org/wiki/Q187874',
 'display': {'label': {'value': 'Plovdiv Province', 'language': 'en'},
  'description': {'value': 'oblast of Bulgaria', 'language': 'en'}},
 'label': 'Plovdiv Province',
 'description': 'oblast of Bulgaria',
 'match': {'type': 'label', 'language': 'en', 'text': 'Plovdiv Province'}}

**Now that we established that this works, let's scale it up**

In [65]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "CTS2000-class-notebook/1.0"}

def get_qid(name):
    """Search Wikidata for a name. Returns (QID, description, status)."""
    params = {"action": "wbsearchentities", "search": name,
              "language": "en", "format": "json"}
    try:
        response = requests.get(WIKIDATA_API, params=params, headers=HEADERS)
        if response.status_code != 200:
            return None, None, f"error {response.status_code}"
        results = response.json()["search"]
        if not results:
            return None, None, "no results"
        return results[0]["id"], results[0].get("description"), "ok"
    except Exception as e:                      # anything unexpected: save it, keep going
        return None, None, f"error: {type(e).__name__}"

Let's unpack this:

This function does exactly what we did for Plovdiv, but for any name we give it. It also adds a few safety checks, because when we send dozens of requests, some of them will go wrong.

`def get_qid(name):` -> we define a function that takes one input, a place name, so we can reuse the same steps for every place on our list.

`params = {...}` -> the same search settings we used for Plovdiv. The only difference is that `"search"` is now `name`, whatever place we pass in, instead of `"Plovdiv"`.

`try:` and `except Exception as e:` -> `try` means "attempt everything in this block". If anything inside it fails (the internet drops, Wikidata sends back something unexpected), Python jumps to `except` instead of crashing. Without this, one bad request would stop the whole loop, and we'd lose our progress on every place after it.

`response = requests.get(WIKIDATA_API, params=params, headers=HEADERS)` -> the GET request, same as before. `WIKIDATA_API` and `HEADERS` are written in capitals because they are **constants**: values we set once and never change.

`if response.status_code != 200:` -> every response comes with a **status code**. `200` means "OK, here is your data". Anything else means something went wrong, so we stop here and return the code, for example `"error 429"`.

`results = response.json()["search"]` -> we turn the response into JSON and keep the list of search results, just like `data["search"]` for Plovdiv.

`if not results:` -> an empty list counts as `False` in Python, so this checks whether Wikidata found nothing. If so, we return `"no results"`.

`return results[0]["id"], results[0].get("description"), "ok"` -> if everything worked, we return three things from the **first** result: its QID, its description and the status `"ok"`. We use `.get("description")` instead of `["description"]` because some Wikidata entries have no description. `.get()` returns `None` when a key is missing, while `["description"]` would cause an error.

`f"error: {type(e).__name__}"` -> if something unexpected went wrong, we save the name of the error (like `ConnectionError`) as our status, so we can look at it later.

Notice that the function **always** returns three values, whether it succeeds or fails. When it fails, the QID and description are `None`. This is what lets us write `qid, description, status = get_qid(...)` in the loop below, and later filter for every place whose status isn't `"ok"`.

In [ ]:
# let's test our function

get_qid("Osijek")

In [ ]:
import time

for i, row in place_names[place_names["QID"].isna()].iterrows():
    qid, description, status = get_qid(row["modern_name"])
    place_names.loc[i, "QID"] = qid
    place_names.loc[i, "description"] = description
    place_names.loc[i, "status"] = status
    time.sleep(1) # be polite: don't hammer the server

place_names


In [ ]:
place_names[place_names["status"] != "ok"]


In [ ]:
place_names[place_names["status"].str.contains('error')]

|    | place  | modern_name   | QID | description | status    |
|----|--------|---------------|-----|-------------|-----------|
| 30 | Cairo  | Cairo         | NaN | NaN         | error 429 |
| 31 | Servia | Serbia        | NaN | NaN         | error 429 |
| 32 | Hebrus | Maritsa River | NaN | NaN         | error 429 |


Error 429 means that we sent in *too many requests* Wikidata is rate-limiting us.

Let's wait a bit and call our function only on these.

In [ ]:
for i, row in place_names[place_names["status"].str.contains('error')].iterrows():
    qid, description, status = get_qid(row["modern_name"])
    place_names.loc[i, "QID"] = qid
    place_names.loc[i, "description"] = description
    place_names.loc[i, "status"] = status
    time.sleep(1) # be polite: don't hammer the server


In [ ]:
place_names

Does anything stand out to you?

In [ ]:
place_names.loc[12, ["place", "description"]]

Europe is tagged as a *"database of biomedical research"*

**(How) Do we fix this?**

1. We can go fix this **manually** and see if there is a Wikidata entry for Europe, the continent (hint, there is: https://www.wikidata.org/wiki/Q46)

2. We can see if this was an issue resulting from our code.

In [ ]:
params = {"action": "wbsearchentities", "search": "Europe",
          "language": "en", "format": "json", "limit": 10}
results = requests.get(WIKIDATA_API, params=params, headers=HEADERS).json()["search"]

pd.DataFrame([{"QID": r["id"], "label": r["label"], "description": r.get("description")}
              for r in results])


Our code only saved the first result, which in this case resulted in a mismatch. Keeping the first result was a shorthand and most places in this list didn't have more than one result to begin with.

3. We can discard it.

Not because of the QID but maybe we should not keep *Europe* and *Asia* for that matter in our place_names list. They are continents and continents are too large to be a single point on our map.

There are merits to keeping continents on a placenames list but this is a decision that needs to be legitimized within a research project. We will talk more about this later but for now, we will discard the two continents.

In [ ]:
place_names = place_names[~place_names["place"].isin(["Europe", "Asia"])].reset_index(drop=True)
place_names

### Coordinates

We can also get coordinates in the same way using the Wikidata API. We just need to modify our function a little.

Why didn't we do this in one pass?

Because these are actually two separate requests, so there is no harm in doing them in two steps. Besides we wanted to make sure that the matches from place names are indeed matching to the places that we want them to. Hence the descriptions.

**Let's test this on Plovdiv again**

In [ ]:
params = {
    "action": "wbgetentities",   # "give me the data for these items"
    "ids": "Q459",               # Plovdiv
    "props": "claims",           # claims = the facts about the item
    "format": "json",
}

data = requests.get(WIKIDATA_API, params=params, headers=HEADERS).json()

# Follow the path down to the coordinates
coords = data["entities"]["Q459"]["claims"]["P625"][0]["mainsnak"]["datavalue"]["value"]
print(coords["latitude"], coords["longitude"])


Now, let's learn a different aspect of the Wikidata API: batches

In [ ]:
qids = place_names["QID"].dropna().tolist()

params = {"action": "wbgetentities", "ids": "|".join(qids),
          "props": "claims", "format": "json"}
entities = requests.get(WIKIDATA_API, params=params, headers=HEADERS).json()["entities"]

def get_coords(qid):
    """Pull latitude and longitude out of one entity. Returns (None, None) if it has none."""
    claims = entities.get(qid, {}).get("claims", {})
    if "P625" not in claims:
        return None, None
    value = claims["P625"][0]["mainsnak"]["datavalue"]["value"]
    return value["latitude"], value["longitude"]


In [ ]:
# let's call this on our df

place_names[["latitude", "longitude"]] = place_names["QID"].apply(
    lambda q: pd.Series(get_coords(q))
)
place_names

In [ ]:
place_names[place_names["latitude"].isna()]

In [ ]:
# Let's save our coordinates and QIDs
# should drop the status col

place_names.to_csv("../../data/montagu/places_for_mapping.csv")

In [66]:
place_names = pd.read_csv("../../data/montagu/places_for_mapping.csv")
place_names.head()

,Unnamed: 0,place,modern_name,QID,description,status,latitude,longitude
0,0,Vienna,Vienna,Q1741,NaN,NaN,48.208333,16.372500
1,1,Constantinople,Istanbul,Q16869,NaN,NaN,41.012240,28.976018
2,2,England,England,Q21,"country in north-west Europe, part of the Unit...",NaN,53.000000,-1.000000
3,3,London,London,Q84,capital and largest city of England and the Un...,NaN,51.507222,-0.127500
4,4,Paris,Paris,Q90,NaN,NaN,48.856667,2.352222


Where were we before this interlude?

Yes, mapping

### A mini exercise in mapping

*What do we need for mapping?*

**Plotly** is a Python library for making interactive charts and maps that you can hover over, zoom into and explore directly in a notebook or web browser.

To display Plotly maps inside the notebook, we need to install one more library, `nbformat`, in our environment.

1. **Open a terminal.** Terminal > New Terminal
2. **Activate our class environment:** `conda activate cts2000`
You should now see `(cts2000)` at the start of the line.
3. **Install the library:** `pip install nbformat`
Wait until you see `Successfully installed nbformat`.
4. **Restart the notebook kernel.** In VS Code, click **Restart** at the top of the notebook. Then re-run the cells from the top.

In [67]:
import plotly.express as px

# bigger dot = more mentions in the letters
place_names["mentions"] = place_names["place"].map(place_counts_updated)

fig = px.scatter_geo(
    place_names,
    lat="latitude",
    lon="longitude",
    hover_name="place",                       # Montagu's name shows when you hover
    hover_data=["modern_name", "mentions"],
    size="mentions",
    #scope="europe",                           # zoom to Europe
)
fig.update_geos(lataxis_range=[25, 60], lonaxis_range=[-10, 40]) # based on our datapoint
fig.show()


Good enough for a first pass, but not good enough for a final presentation. Still a great first attempt at mapping from historical placenames

### Where she wrote from vs. what she wrote about

Beyond counting the most frequent mentions, placenames and NER can offer other useful insights into our letters. One idea is the following:

**Is there a relationship between the places Montagu sends her letters from and the places she writes about in her letters?**

Our `montagu_letters_v1.csv` records where each letter was sent from, in Montagu's own spelling. We had dropped in when we transitioned to analysing the bodies of the letters. Our `locations.csv` which we already used in the Wikidata API exercise above, has the unique places that Montagu sent letters from, but under modern names. This is because I created that csv to work with the QID and coordinates exercise. I identified the modern place name for each of the locations, which means that we can revert it semi-automatically. We just need the same historical-to-modern mapping we used for the places she mentions.

In [76]:
# Step 1: load the sending locations from our first version of the letters
letter_info = pd.read_csv("../../data/montagu/montagu_letters_v1.csv")[["filename", "location"]]

# clean them: remove underscores and notes like "(sic)", then extra spaces
letter_info["location_clean"] = (letter_info["location"]
                                 .str.replace("_", "")
                                 .str.replace(r"\(.*\)", "", regex=True)
                                 .str.strip())

letter_info["location_clean"].value_counts()

location_clean
Vienna                    11
Adrianople                 9
Hanover                    2
Constantinople             2
Belgrade Village           2
Pera of Constantinople     2
Lyons                      2
Paris                      2
Dover                      2
Rotterdam                  1
Hague                      1
Nimeguen                   1
Cologn                     1
Nuremberg                  1
Ratisbon                   1
Prague                     1
Leipzig                    1
Brunswick                  1
Blankenburg                1
Peterwaradin               1
Belgrade                   1
Adrianopolis               1
Pera                       1
Tunis                      1
Genoa                      1
Turin                      1
Name: count, dtype: int64

In [77]:
# Step 2: Montagu's spelling -> the modern name used in locations.csv
historical_to_modern = {
    "Constantinople": "Istanbul",
    "Pera of Constantinople": "Pera",
    "Adrianople": "Edirne",
    "Adrianopolis": "Edirne",          # she spells it two ways!
    "Peterwaradin": "Petrovaradin",
    "Blankenburg": "Blankenburg am Harz",
    "Brunswick": "Braunschweig",
    "Cologn": "Cologne",
}

letter_info["location_name"] = letter_info["location_clean"].map(lambda p: historical_to_modern.get(p, p))

letter_info.head()

,filename,location,location_clean,location_name
0,letter_1.txt,_Rotterdam,Rotterdam,Rotterdam
1,letter_2.txt,_Hague,Hague,Hague
2,letter_3.txt,_Nimeguen,Nimeguen,Nimeguen
3,letter_4.txt,_Cologn (sic),Cologn,Cologne
4,letter_5.txt,_Nuremberg,Nuremberg,Nuremberg


In [ ]:
# Step 3: add the sending location and its coordinates to each letter
letter_info = letter_info.merge(locations, on="location_name", how="left")
montagu = montagu.merge(letter_info[["filename", "location_name", "latitude", "longitude"]],
                        on="filename", how="left")

Let's unpack this:

This cell has two **merges**. A merge joins two tables by matching up the values in a column they share, a bit like `VLOOKUP` in Excel.

`letter_info.merge(locations, on="location_name", how="left")` -> the first merge adds coordinates to each letter. Both tables have a `location_name` column, so pandas matches them up: every letter sent from *Vienna* gets Vienna's QID, latitude and longitude from `locations.csv`. This only works because we mapped Montagu's spellings to the modern names in Step 2. Without that, *Adrianople* in our letters would never match *Edirne* in `locations.csv`.

`on="location_name"` -> the column to match on. It has to exist in both tables, and the values have to be **exactly** the same: *Pera of Constantinople* and *Pera* do not match.

`how="left"` -> keep **every** row from the left table (`letter_info`), even if it has no match on the right. If we left this out, pandas would use `how="inner"` and keep only the rows that match in both tables.

Notice that this is a **many-to-one** merge: Vienna appears once in `locations.csv` but eleven times in `letter_info`, once for each letter she wrote there. The merge copies Vienna's coordinates to all eleven.

---

`montagu.merge(letter_info[["filename", "location_name", "latitude", "longitude"]], on="filename", how="left")` -> the second merge adds this information to our main `montagu` table. This time we match on `filename`, because that's what the two tables share: each letter has one row in each.

`letter_info[["filename", "location_name", "latitude", "longitude"]]` -> we only bring over the columns we need. Otherwise `montagu` would also get `location`, `location_clean`, `location_id` and `location_wikidata`, which we won't use.

In [75]:
locations.head()

,location_id,location_name,location_wikidata,latitude,longitude
0,LOC-001,Rotterdam,Q34370,51.920000,4.480000
1,LOC-002,Hague,Q36600,52.080000,4.310000
2,LOC-003,Nimeguen,Q47887,51.847500,5.862500
3,LOC-004,Cologne,Q365,50.942222,6.957778
4,LOC-005,Nuremberg,Q2090,49.453889,11.077500


In [79]:
letter_info.head()

,filename,location,location_clean,location_name,location_id,location_wikidata,latitude,longitude
0,letter_1.txt,_Rotterdam,Rotterdam,Rotterdam,LOC-001,Q34370,51.920000,4.480000
1,letter_2.txt,_Hague,Hague,Hague,LOC-002,Q36600,52.080000,4.310000
2,letter_3.txt,_Nimeguen,Nimeguen,Nimeguen,LOC-003,Q47887,51.847500,5.862500
3,letter_4.txt,_Cologn (sic),Cologn,Cologne,LOC-004,Q365,50.942222,6.957778
4,letter_5.txt,_Nuremberg,Nuremberg,Nuremberg,LOC-005,Q2090,49.453889,11.077500


In [73]:
montagu.head()

,filename,title,body,entities,sentences,num_sents,words,lemmas,pos_tags,xpos_tags,morph_feats,dependencies,sentiment,mean_sentiment,places,places_fixed,location_name,latitude,longitude
0,letter_1.txt,﻿LETTER 1.,"I FLATTER, myself, dear sister, that I shall g...","[(two days, DATE), (Rotterdam, GPE), (Helvoets...","['I FLATTER, myself, dear sister, that I shall...",18,"['I', 'FLATTER', ',', 'myself', ',', 'dear', '...","['I', 'flatter', ',', 'myself', ',', 'dear', '...","[('I', 'PRON'), ('FLATTER', 'VERB'), (',', 'PU...","[('I', 'PRP'), ('FLATTER', 'VBP'), (',', ','),...","[('I', 'Case=Nom|Number=Sing|Person=1|PronType...","[('I', 'nsubj', 'FLATTER'), ('FLATTER', 'root'...","[1, 2, 1, 1, 2, 0, 1, 1, 1, 2, 1, 0, 2, 2, 2, ...",1.222222,"{'Rotterdam': 3, 'Helvoetsluys': 1, 'England':...","{'Rotterdam': 3, 'Helvoetsluys': 1, 'England':...",Rotterdam,51.920000,4.480000
1,letter_2.txt,LET. II,"I MAKE haste to tell you, dear Madam, that, af...","[(Madam, PERSON), (Holland, GPE), (twenty, CAR...","['I MAKE haste to tell you, dear Madam, that, ...",12,"['I', 'MAKE', 'haste', 'to', 'tell', 'you', ',...","['I', 'make', 'haste', 'to', 'tell', 'you', ',...","[('I', 'PRON'), ('MAKE', 'VERB'), ('haste', 'N...","[('I', 'PRP'), ('MAKE', 'VBP'), ('haste', 'NN'...","[('I', 'Case=Nom|Number=Sing|Person=1|PronType...","[('I', 'nsubj', 'MAKE'), ('MAKE', 'root', 'ROO...","[2, 1, 2, 2, 2, 2, 2, 1, 1, 0, 1, 2]",1.500000,"{'Holland': 1, 'London': 1, 'India': 1}","{'Holland': 1, 'London': 1, 'India': 1}",Hague,52.080000,4.310000
2,letter_3.txt,LET. III,"I AM extremely sorry, my dear S. that your fea...","[(S., PERSON), (Nottingham, GPE), (two, CARDIN...","['I AM extremely sorry, my dear S. that your f...",18,"['I', 'AM', 'extremely', 'sorry', ',', 'my', '...","['I', 'be', 'extremely', 'sorry', ',', 'my', '...","[('I', 'PRON'), ('AM', 'AUX'), ('extremely', '...","[('I', 'PRP'), ('AM', 'VBP'), ('extremely', 'R...","[('I', 'Case=Nom|Number=Sing|Person=1|PronType...","[('I', 'nsubj', 'sorry'), ('AM', 'cop', 'sorry...","[1, 1, 1, 1, 1, 2, 0, 2, 2, 2, 1, 1, 1, 1, 2, ...",1.166667,"{'Nottingham': 3, 'Trentfield': 1, 'Adboulton'...","{'Nottingham': 3, 'Trentfield': 1, 'Adboulton'...",Nimeguen,51.847500,5.862500
3,letter_4.txt,LET. IV.,IF my lady ---- could have any notion of the f...,"[(two last days, DATE), (Nimeguen, GPE), (Rein...",['IF my lady ---- could have any notion of the...,12,"['IF', 'my', 'lady', '----', 'could', 'have', ...","['if', 'my', 'lady', '----', 'could', 'have', ...","[('IF', 'SCONJ'), ('my', 'PRON'), ('lady', 'NO...","[('IF', 'IN'), ('my', 'PRP$'), ('lady', 'NN'),...","[('IF', None), ('my', 'Case=Gen|Number=Sing|Pe...","[('IF', 'mark', 'have'), ('my', 'nmod:poss', '...","[0, 1, 0, 0, 2, 0, 1, 2, 1, 1, 1, 1]",0.833333,"{'Nimeguen': 1, 'Stamel': 1, 'St Ursula's': 1,...","{'Nimeguen': 1, 'Stamel': 1, 'St Ursula's': 1,...",Cologne,50.942222,6.957778
4,letter_5.txt,LET. V.,"AFTER five days travelling post, I could not s...","[(five days, DATE), (Germany, GPE), (Cologn, G...","['AFTER five days travelling post, I could not...",19,"['AFTER', 'five', 'days', 'travelling', 'post'...","['after', 'five', 'day', 'travel', 'post', ','...","[('AFTER', 'ADP'), ('five', 'NUM'), ('days', '...","[('AFTER', 'IN'), ('five', 'CD'), ('days', 'NN...","[('AFTER', None), ('five', 'NumForm=Word|NumTy...","[('AFTER', 'case', 'post'), ('five', 'nummod',...","[0, 0, 1, 1, 1, 2, 0, 0, 1, 2, 0, 1, 1, 1, 1, ...",0.789474,"{'Germany': 2, 'Cologn': 1, 'Frankfort': 1, 'W...","{'Germany': 2, 'Cologn': 1, 'Frankfort': 1, 'W...",Nuremberg,49.453889,11.077500


In [80]:
montagu["location_name"].value_counts()

location_name
Vienna                 10
Edirne                 10
Pera                    3
Hanover                 2
Istanbul                2
Belgrade Village        2
Lyons                   2
Paris                   2
Dover                   2
Rotterdam               1
Hague                   1
Nimeguen                1
Cologne                 1
Nuremberg               1
Ratisbon                1
Prague                  1
Leipzig                 1
Braunschweig            1
Blankenburg am Harz     1
Petrovaradin            1
Belgrade                1
Tunis                   1
Genoa                   1
Turin                   1
Name: count, dtype: int64

In [82]:
montagu.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   filename        52 non-null     str    
 1   title           52 non-null     str    
 2   body            52 non-null     str    
 3   entities        52 non-null     object 
 4   sentences       52 non-null     str    
 5   num_sents       52 non-null     int64  
 6   words           52 non-null     str    
 7   lemmas          52 non-null     str    
 8   pos_tags        52 non-null     str    
 9   xpos_tags       52 non-null     str    
 10  morph_feats     52 non-null     str    
 11  dependencies    52 non-null     str    
 12  sentiment       52 non-null     str    
 13  mean_sentiment  52 non-null     float64
 14  places          52 non-null     object 
 15  places_fixed    52 non-null     object 
 16  location_name   50 non-null     str    
 17  latitude        50 non-null     float64
 18  lon

In [86]:
# Let's see which letters have no origins

montagu[montagu["location_name"].isna()][["filename", "title"]]

,filename,title
40,letter_41.txt,LET. XLI.
41,letter_42.txt,LET. XLII.


Letters 41 and 42 do not have origins mentioned in the letters themselves. But some sleuthing can lead us to an educated guess

- Letter 41 opens with “Since my last, I have staid quietly at Constantinople” but later in the same paragraph she starts talking about Pera, where she is residing while in Constantinople.
- Letter 42 opens with "I AM now preparing to leave Constantinople" but again later she is writing about the affairs in her neighborhood of Pera. 

Montagu generally uses Pera and Constantinople interchangably even though she knows and describes the administrative and cultural differenes between the two. Historically Constantinople refers to what is the Topkapi district today and Pera, now a neighborhood of Istanbul, was then a settlement across the Golden Horne which housed the European diplomatic envoys. 

In this case, we will make a practical decision and assume that she is writing from her home in Pera and edit the empty locations accordingly.

In [87]:
# letters 41 and 42 have no location, but the letters themselves place her in Pera
pera = locations[locations["location_name"] == "Pera"].iloc[0]

missing = montagu["filename"].isin(["letter_41.txt", "letter_42.txt"])
montagu.loc[missing, ["location_name", "latitude", "longitude"]] = ["Pera", pera["latitude"], pera["longitude"]]

# check
montagu.loc[missing, ["filename", "location_name", "latitude", "longitude"]]

,filename,location_name,latitude,longitude
40,letter_41.txt,Pera,41.031944,28.976111
41,letter_42.txt,Pera,41.031944,28.976111


In [88]:
montagu.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   filename        52 non-null     str    
 1   title           52 non-null     str    
 2   body            52 non-null     str    
 3   entities        52 non-null     object 
 4   sentences       52 non-null     str    
 5   num_sents       52 non-null     int64  
 6   words           52 non-null     str    
 7   lemmas          52 non-null     str    
 8   pos_tags        52 non-null     str    
 9   xpos_tags       52 non-null     str    
 10  morph_feats     52 non-null     str    
 11  dependencies    52 non-null     str    
 12  sentiment       52 non-null     str    
 13  mean_sentiment  52 non-null     float64
 14  places          52 non-null     object 
 15  places_fixed    52 non-null     object 
 16  location_name   52 non-null     str    
 17  latitude        52 non-null     float64
 18  lon

Now, let's return to our question: **Is there a relationship between the places Montagu sends her letters from and the places she writes about in her letters?**

Let's make a table: for each place she wrote from, how often does she mention each place on our top places list?

In [89]:
# Step 1: one row for every place mentioned in every letter, with where that letter was sent from
to_modern = dict(zip(place_names["place"], place_names["modern_name"]))

rows = []
for i, letter in montagu.dropna(subset=["location_name"]).iterrows():
    for place, count in letter["places"].items():
        if place in to_modern:                       # only places on our map list
            rows.append({"sent_from": letter["location_name"],
                         "mentioned": to_modern[place],
                         "count": count})

mentions = pd.DataFrame(rows)
mentions.head()

,sent_from,mentioned,count
0,Rotterdam,England,1
1,Rotterdam,London,2
2,Rotterdam,Netherlands,1
3,Hague,Netherlands,1
4,Hague,London,1


In [90]:
# Step 2: turn it into a table, sending locations as rows, mentioned places as columns
table = mentions.pivot_table(index="sent_from", columns="mentioned",
                             values="count", aggfunc="sum", fill_value=0)

# put both in the order they first appear in the letters, so roughly the order of her journey
route = mentions["sent_from"].drop_duplicates()
first_mentioned = mentions["mentioned"].drop_duplicates()
table = table.loc[route, first_mentioned]

table

mentioned,England,London,Netherlands,Germany,Hanover,Vienna,Danube,Genoa,Austria,Bohemia,...,Rome,Istanbul,Cairo,Ottoman Empire,Italy,France,Pera,Paris,Tunis,Turin
sent_from,,,,,,,,,,,,,,,,,,,,,
Rotterdam,1,2,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Hague,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Nimeguen,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cologne,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Nuremberg,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Ratisbon,0,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Vienna,5,2,0,0,1,19,4,1,2,1,...,0,0,0,0,0,0,0,0,0,0
Prague,0,1,0,2,0,2,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
Leipzig,0,2,0,2,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
fig = px.imshow(
    table,
    color_continuous_scale="Purples",
    range_color=[0, 8],      # anything above 8 is shown as the darkest colour
    aspect="auto",
    labels=dict(x="place mentioned", y="sent from", color="mentions"),
)
fig.update_xaxes(tickangle=45, tickmode="array", tickvals=table.columns, ticktext=table.columns)
fig.update_yaxes(tickmode="array", tickvals=table.index, ticktext=table.index)
fig.update_layout(height=900)
fig.show()